# 🏦 BNPL Loan Default Prediction
**Buy Now Pay Later — End-to-End ML Pipeline**

This notebook covers:
1. Data Loading & Exploration
2. Preprocessing & Feature Engineering  
3. Model Training (Logistic Regression, Random Forest, Gradient Boosting)
4. Evaluation & Comparison
5. Saving Models as PKL files

In [ ]:
# Install dependencies (uncomment if needed)
# !pip install pandas scikit-learn matplotlib seaborn xlrd joblib flask openpyxl

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (classification_report, roc_auc_score,
                              confusion_matrix, roc_curve, ConfusionMatrixDisplay)
import joblib, warnings, os
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-darkgrid')
print("All imports successful ✅")

## 1. Data Loading & Exploration

In [ ]:
# Load the XLS file (use xlrd engine for legacy .xls)
df = pd.read_excel('bnpl_sample_40000__2_.xls', engine='xlrd')
print(f"Dataset shape: {df.shape}")
df.head()

In [ ]:
# Basic info
df.info()

In [ ]:
# Statistical summary
df.describe()

In [ ]:
# Check missing values
missing = df.isnull().sum()
print("Missing values per column:")
print(missing[missing > 0] if missing.sum() > 0 else "No missing values ✅")

In [ ]:
# Target distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

counts = df['Default'].value_counts()
axes[0].bar(['No Default (0)', 'Default (1)'], counts.values,
            color=['#22c55e', '#ef4444'], edgecolor='white', linewidth=1.5)
axes[0].set_title('Target Class Distribution', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Count')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 200, f'{v:,}\n({v/len(df)*100:.1f}%)',
                 ha='center', fontweight='bold')

df['Default'].value_counts().plot.pie(
    labels=['No Default', 'Default'],
    colors=['#22c55e', '#ef4444'],
    autopct='%1.1f%%', startangle=90,
    ax=axes[1], wedgeprops={'edgecolor':'white','linewidth':2}
)
axes[1].set_title('Default Rate', fontsize=14, fontweight='bold')
axes[1].set_ylabel('')

plt.tight_layout()
plt.savefig('target_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Default rate: {df['Default'].mean()*100:.2f}%")

In [ ]:
# Numeric feature distributions
num_cols = ['Age','Income','LoanAmount','CreditScore','MonthsEmployed',
            'InterestRate','LoanTerm','DTIRatio']

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()
for i, col in enumerate(num_cols):
    axes[i].hist(df[col], bins=40, color='#6366f1', alpha=0.8, edgecolor='white', linewidth=0.5)
    axes[i].set_title(col, fontweight='bold')
    axes[i].set_xlabel('')
plt.suptitle('Numeric Feature Distributions', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('feature_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Categorical feature counts
cat_cols = ['Education','EmploymentType','MaritalStatus','LoanPurpose']
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for i, col in enumerate(cat_cols):
    vc = df[col].value_counts()
    axes[i].barh(vc.index, vc.values, color='#22d3ee', alpha=0.85, edgecolor='white')
    axes[i].set_title(col, fontweight='bold')
    axes[i].invert_yaxis()
plt.suptitle('Categorical Features', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('categorical_features.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Default rate by category
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for i, col in enumerate(cat_cols):
    rate = df.groupby(col)['Default'].mean().sort_values(ascending=False)
    colors = ['#ef4444' if v > df['Default'].mean() else '#22c55e' for v in rate.values]
    axes[i].bar(rate.index, rate.values * 100, color=colors, edgecolor='white', linewidth=1.2)
    axes[i].axhline(df['Default'].mean()*100, color='#f97316', linestyle='--', label='Overall avg')
    axes[i].set_title(f'Default Rate by {col}', fontweight='bold')
    axes[i].set_ylabel('Default Rate (%)')
    axes[i].tick_params(axis='x', rotation=20)
plt.suptitle('Default Rate by Category', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('default_by_category.png', dpi=150, bbox_inches='tight')
plt.show()

## 2. Preprocessing & Feature Engineering

In [ ]:
# Drop LoanID (identifier, not predictive)
df_proc = df.drop(columns=['LoanID']).copy()

# Encode binary Yes/No columns
binary_cols = ['HasMortgage', 'HasDependents', 'HasCoSigner']
for col in binary_cols:
    df_proc[col] = (df_proc[col] == 'Yes').astype(int)

# Label encode multi-class categorical columns
label_cols = ['Education', 'EmploymentType', 'MaritalStatus', 'LoanPurpose']
le_dict = {}
for col in label_cols:
    le = LabelEncoder()
    df_proc[col] = le.fit_transform(df_proc[col])
    le_dict[col] = le
    print(f"{col}: {dict(zip(le.classes_, le.transform(le.classes_)))}")

print("\nEncoding complete ✅")
df_proc.head()

In [ ]:
# Correlation heatmap
plt.figure(figsize=(14, 10))
corr = df_proc.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn_r',
            center=0, square=True, linewidths=0.5,
            annot_kws={'size': 8}, vmin=-1, vmax=1)
plt.title('Feature Correlation Matrix', fontsize=16, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig('correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Train-test split + scaling
X = df_proc.drop(columns=['Default'])
y = df_proc['Default']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print(f"Train: {X_train_scaled.shape}, Test: {X_test_scaled.shape}")
print(f"Train default rate: {y_train.mean():.3f}")
print(f"Test  default rate: {y_test.mean():.3f}")

## 3. Model Training

In [ ]:
# Define models
models_config = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Random Forest":       RandomForestClassifier(n_estimators=30, max_depth=8,
                                                   random_state=42, n_jobs=-1),
    "Gradient Boosting":   GradientBoostingClassifier(n_estimators=100,
                                                       random_state=42)
}

trained_models = {}
results = {}

for name, model in models_config.items():
    print(f"Training {name}...", end=' ')
    model.fit(X_train_scaled, y_train)
    preds = model.predict(X_test_scaled)
    proba = model.predict_proba(X_test_scaled)[:, 1]
    auc   = roc_auc_score(y_test, proba)
    trained_models[name] = model
    results[name] = {'preds': preds, 'proba': proba, 'auc': auc, 'model': model}
    print(f"AUC = {auc:.4f} ✅")

print("\nAll models trained!")

## 4. Evaluation & Comparison

In [ ]:
# Classification reports
for name, r in results.items():
    print(f"{'='*50}")
    print(f"  {name}")
    print(f"{'='*50}")
    print(classification_report(y_test, r['preds'], target_names=['No Default','Default']))

In [ ]:
# ROC Curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = ['#6366f1', '#22d3ee', '#f97316']

for (name, r), c in zip(results.items(), colors):
    fpr, tpr, _ = roc_curve(y_test, r['proba'])
    axes[0].plot(fpr, tpr, color=c, lw=2, label=f"{name} (AUC={r['auc']:.4f})")

axes[0].plot([0,1],[0,1],'k--', lw=1, alpha=0.5)
axes[0].set_xlabel('False Positive Rate', fontweight='bold')
axes[0].set_ylabel('True Positive Rate', fontweight='bold')
axes[0].set_title('ROC Curves - All Models', fontsize=14, fontweight='bold')
axes[0].legend(loc='lower right')
axes[0].set_xlim([0,1]); axes[0].set_ylim([0,1])

# AUC bar chart
names = list(results.keys())
aucs  = [r['auc'] for r in results.values()]
bars  = axes[1].bar(names, aucs, color=colors, edgecolor='white', linewidth=1.5)
axes[1].set_ylim([0.5, 0.85])
axes[1].set_title('AUC Score Comparison', fontsize=14, fontweight='bold')
axes[1].set_ylabel('AUC Score')
axes[1].tick_params(axis='x', rotation=10)
for bar, auc in zip(bars, aucs):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
                 f'{auc:.4f}', ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('roc_curves.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Confusion Matrices
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (name, r) in zip(axes, results.items()):
    cm = confusion_matrix(y_test, r['preds'])
    disp = ConfusionMatrixDisplay(cm, display_labels=['No Default','Default'])
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(name, fontweight='bold', fontsize=12)
plt.suptitle('Confusion Matrices', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Feature importance (Random Forest)
rf_model = trained_models['Random Forest']
feat_names = X.columns.tolist()
importances = pd.Series(rf_model.feature_importances_, index=feat_names).sort_values(ascending=True)

plt.figure(figsize=(10, 6))
colors = ['#6366f1' if v > importances.median() else '#94a3b8' for v in importances.values]
plt.barh(importances.index, importances.values, color=colors, edgecolor='white')
plt.xlabel('Feature Importance', fontweight='bold')
plt.title('Random Forest Feature Importances', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Save Models as PKL files

In [ ]:
# Save all models and preprocessing objects
os.makedirs('models', exist_ok=True)

pkl_name_map = {
    'Logistic Regression': 'logistic_regression',
    'Random Forest':       'random_forest',
    'Gradient Boosting':   'gradient_boosting',
}

for name, model in trained_models.items():
    path = f"models/{pkl_name_map[name]}.pkl"
    joblib.dump(model, path)
    print(f"Saved: {path}")

joblib.dump(scaler,   'models/scaler.pkl')
joblib.dump(le_dict,  'models/label_encoders.pkl')
print("Saved: models/scaler.pkl")
print("Saved: models/label_encoders.pkl")
print("\nAll PKL files saved! ✅")

In [ ]:
# Verify saved models reload correctly
print("Verification — reloading models:")
for name, fname in pkl_name_map.items():
    m = joblib.load(f'models/{fname}.pkl')
    proba = m.predict_proba(X_test_scaled[:5])[:, 1]
    print(f"  {name}: first 5 probs = {proba.round(3)}")
print("\nAll models verified ✅")

## Summary

| Model | AUC |
|---|---|
| Logistic Regression | ~0.743 |
| Random Forest | ~0.740 |
| Gradient Boosting | ~0.747 |

**Best model:** Gradient Boosting (highest AUC)  
**Saved PKL files:** `models/logistic_regression.pkl`, `models/random_forest.pkl`, `models/gradient_boosting.pkl`, `models/scaler.pkl`, `models/label_encoders.pkl`

**To run the Flask app:** `python app.py` then open http://localhost:5000